# BrainLearn EEG cohort replay

Research-only, experimental export. The project does not include EEG data.


## 1. Setup
Install `environment.lock` and set the local BIDS path below.

In [ ]:
from pathlib import Path

from workflow import inspect_inputs, load_config, run_workflow, show_recorded_qc

config = load_config()
dataset_root = Path("./dataset").resolve()  # change to your local BIDS copy

## 2. Inspect inputs
Check the source checksums, channel/rate compatibility, and event labels before processing.

In [ ]:
inspection = inspect_inputs(config, dataset_root)
inspection

## 3. Quality-control decisions
Bad-channel choices below are recorded decisions bound to the source hashes. Review them and explicitly acknowledge before reuse. ICA component choices are not reused: ICA is refitted, so inspect the new fit and make a fresh decision.

In [ ]:
show_recorded_qc(config)
reuse_recorded_bad_channel_qc = False  # set True only after reviewing the listed decisions

## 4. Processing
Running this cell asks for a fresh reviewer decision after each ICA fit. It will stop if the review is rejected or missing.

In [ ]:
if not reuse_recorded_bad_channel_qc:
    raise RuntimeError(
        "Review the saved bad-channel decisions and explicitly set the acknowledgement above."
    )
results = run_workflow(
    config, dataset_root, reuse_recorded_bad_channel_qc=True, show_ica_diagnostics=True
)
results

## 5. Results
Review participant-level ERP/PSD summaries, denominators, failures, and the generated HTML report.

In [ ]:
import json

print((Path(results["output_dir"]) / "report.html").read_text(encoding="utf-8")[:4000])
json.loads((Path(results["output_dir"]) / "summary.json").read_text(encoding="utf-8"))